In [10]:
import ast

with open("examples/lu_core.dsl.py","r") as f:
    source = f.read()

tree = ast.parse(source)
print(ast.dump(tree,indent=2))

Module(
  body=[
    FunctionDef(
      name='lu_core',
      args=arguments(
        args=[
          arg(arg='par_n')]),
      body=[
        Assign(
          targets=[
            Name(id='f32_1', ctx=Store())],
          value=Constant(value=0)),
        Assign(
          targets=[
            Name(id='f32_2', ctx=Store())],
          value=Constant(value=0)),
        Assign(
          targets=[
            Name(id='f32_3', ctx=Store())],
          value=Constant(value=0)),
        Assign(
          targets=[
            Name(id='u8_i', ctx=Store())],
          value=Constant(value=0)),
        Assign(
          targets=[
            Name(id='u8_j', ctx=Store())],
          value=Constant(value=0)),
        Assign(
          targets=[
            Name(id='u8_k', ctx=Store())],
          value=Constant(value=0)),
        Assign(
          targets=[
            Name(id='u8_m', ctx=Store())],
          value=Constant(value=0)),
        For(
          target=Name(id='u8_j', ctx=Store(

- interpret all the top function parameters (ignore typing) as integer parameters
- all local variables have to be declared and initialized at the top
- treat all local variables as binary unsigned integers, use their name to specify the width of the variable (form of `Xkk_xxxx` where `X` is one character only for noting its sematic type without compiling restriction like `f` for floats, `s` for signed integers and `u` for unsigned intrgers; `kk` is the width of the variable like `32` (currently with no check), and `xxxx` is just a label. when converting to verilog, we retain the full name `Xkk_xxxx` as name of the generated register)
- support arbitrary primitive, and by forcing named parameters when calling them to explicitly link the blocking module's port name
  - can add the optinal primitive metadata list later (to give latency / tying info)
- primitive whose name ends with `_comb` will be converted to a combinational verilog function wrapper

example:

```python
def lu_core(par_n):
    f32_1 = 0
    f32_2 = 0
    f32_3 = 0
    u8_i = 0
    u8_j = 0
    u8_k = 0
    for u8_j in range(par_n):
        for u8_i in range(par_n):
            f32_1 = fetch_A(i=u8_i, j=u8_j)
            f32_1 = neg_comb(v=f32_1)
            u8_m = u8_j if u8_i > u8_j else u8_i 
            for u8_k in range(u8_m):
                f32_2 = fetch_LU(i=u8_i, j=u8_k)
                f32_3 = fetch_LU(i=u8_k, j=u8_j)
                f32_1 = fma(a=f32_2, b=f32_3, c=f32_1)
            if u8_i > u8_j:
                f32_2 = fetch_LU(i=u8_j, j=u8_j)
                f32_1 = div(a=f32_1, b=f32_2)
            f32_1 = neg_comb(v=f32_1)
            store_LU(i=u8_i, j=u8_j, v=f32_1)
```

I've attached the dumped ast tree as a txt file

In [11]:
from dataclasses import asdict
import json

class SetEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, set):
            return list(obj)
        return json.JSONEncoder.default(self, obj)
    
def dump_file(path:str,content:str):
    with open(path, "w") as dump_file:
        dump_file.write(content)
    
def dump_json(path:str,stuff):
    with open(path, "w") as dump_file:
        json.dump(asdict(stuff), dump_file, indent=2, cls=SetEncoder)

In [12]:
from pathlib import Path
from compiler.ast_checker import DSLFrontendChecker
from compiler.ast_to_hir import lower_source
from compiler.hir_to_lir import lower_func

ROOT = Path(".").resolve()
EXAMPLEPATH = ROOT / "examples" / "lu_core.dsl.py"
EXAMPLE = EXAMPLEPATH.read_text()

checker_inst = DSLFrontendChecker()
frontend_info = checker_inst.check_source(EXAMPLE)
dump_json("tmp/frontend_info.dump.json", frontend_info)
func_ir = lower_source(EXAMPLE)
dump_json("tmp/func_ir.dump.json", func_ir)
dump_file("tmp/func_ir.dump.txt", func_ir.__repr__())
func_lir = lower_func(func_ir)
dump_json("tmp/func_lir.dump.json", func_lir)
dump_file("tmp/func_lir.dump.txt", func_lir.__repr__())

In [13]:
from compiler.sim_runtime import (
    PrimitiveModel,
    SimulationHarness,
    run_all,
)


def fetch_A(tb, i, j):
    return tb.state["A"][i][j]


def fetch_LU(tb, i, j):
    return tb.state["LU"][i][j]


def store_LU(tb, i, j, v):
    tb.state["LU"][i][j] = v


def neg_comb(tb, v):
    return -v


def fma(tb, a, b, c):
    return (a * b) + c


def div(tb, a, b):
    return a / b

ROOT = Path(".").resolve()
LU_CORE = (ROOT / "examples" / "lu_core.dsl.py").read_text()

harness = SimulationHarness(
    params={"par_n": 3},
    primitives={
        "fetch_A": PrimitiveModel(name="fetch_A", impl=fetch_A, latency=2),
        "fetch_LU": PrimitiveModel(name="fetch_LU", impl=fetch_LU, latency=2),
        "store_LU": PrimitiveModel(name="store_LU", impl=store_LU, latency=1),
        "neg_comb": PrimitiveModel(name="neg_comb", impl=neg_comb),
        "fma": PrimitiveModel(name="fma", impl=fma, latency=3),
        "div": PrimitiveModel(name="div", impl=div, latency=4),
    },
    initial_state={
        "A": [
            [4.0, 3.0, 2.0],
            [6.0, 3.0, 0.0],
            [2.0, 1.0, 1.0],
        ],
        "LU": [
            [0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0],
        ],
    },
)
report = run_all(LU_CORE, harness)
print(report)

EquivalenceReport[OK]
